# Ledger contributions per layer group, baseline vs ablation

The showcase §3 stacked decomposition (ΔS = χ + quality + interference), but summed
over a chosen layer group only, side by side: left the unablated packed sweep, right
the run with a block's writes zeroed (data/results/ablate_*).

In [ ]:
import os, sys, importlib
if os.path.basename(os.getcwd()) == "analysis":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import experiments_lib as _lib
importlib.reload(_lib)
from analysis.experiments_lib import (plot_layer_contribution, plot_ledger_stack,
                                      grid_start, grid_show, get_model_label)

NB = {'pythia-160m-deduped': 12, 'pythia-410m-deduped': 24, 'pythia-1b-deduped': 16,
      'pythia-6.9b-deduped': 32, 'OLMo-2-0425-1B': 16, 'OLMo-2-1124-7B': 32,
      'nanochat-d12': 12}
CFG = lambda m: 'nanochat_samples' if m == 'nanochat-d12' else 'block_representations_samples'
QUARTERS = {12: ['blk0-2', 'blk3-5', 'blk6-8', 'blk9-11', 'blk3-8'],
            16: ['blk0-3', 'blk4-7', 'blk8-11', 'blk12-15', 'blk4-11'],
            24: ['blk0-5', 'blk6-11', 'blk12-17', 'blk18-23', 'blk6-17'],
            32: ['blk0-7', 'blk8-15', 'blk16-23', 'blk24-31', 'blk8-23']}
CARRIER = {'pythia-1b-deduped': 'blk3', 'pythia-6.9b-deduped': 'blk3-4'}

def ledger_grid(model, rows, cols, title, savedir, figsize=None):
    """Grid of experiments_lib.plot_ledger_stack panels: rows = (label, block list),
    cols = (label, results dir). sharey='row' keeps each row's panels comparable without
    flattening single-layer rows against whole-quarter ones; all_xlabels keeps the token
    axis on every row. emb=False: the embedding line is only meaningful full-depth."""
    grid_start(ncols=len(cols), figsize=figsize or (6.5 * len(cols), 4 * len(rows)),
               sharex=True, sharey='row', all_xlabels=True, emb=False, zero_line=True,
               title=title, savedir=savedir)
    for i, (rlabel, layers) in enumerate(rows):
        for j, (clabel, cfg) in enumerate(cols):
            plot_ledger_stack(model, cfg, blocks=layers, title=f'{clabel} — {rlabel}',
                              ylabel='rank entropy' if j == 0 else None,
                              legend=7 if i == 0 and j == 0 else None)
    grid_show()

def ledger_fig(model, ablate):
    """3x2: rows = penultimate / final layer / last quarter, cols = baseline / ablated."""
    L = NB[model]
    rows = [(f'penultimate layer (blk{L - 2})', [L - 2]),
            (f'final layer (blk{L - 1})', [L - 1]),
            (f'last quarter (blk{3 * L // 4}-{L - 1})', list(range(3 * L // 4, L)))]
    ledger_grid(model, rows, [('baseline', CFG(model)), (f'− {ablate}', f'ablate_{ablate}')],
                f'Ledger contributions per layer group — {model}, {ablate} writes zeroed',
                f'ledger_ablation_layers/{model}_{ablate}')

In [ ]:
# pythia-1b (16 blocks), blk3 (the carrier) ablated. Rows: the penultimate layer's
# contributions (blk14), the final layer's (blk15), the last quarter's (blk12-15).
# Left baseline, right ablated.
for model, blks in CARRIER.items():
    ledger_fig(model, blks)
# FIGURE B2?.?
# Not sure where, but this needs to be in. The pythia 6.9b carrier (blk3-4) is now included
# via the ABLATIONS loop below.

In [ ]:
# The other models: the carrier where one exists (pythia-6.9b: blk3-4), otherwise the
# first-quarter ablation — the early-blocks analog of blk3 (no other single-block runs).
ABLATIONS = {'pythia-160m-deduped': 'blk0-2', 'pythia-410m-deduped': 'blk0-5',
             'pythia-6.9b-deduped': 'blk3-4', 'OLMo-2-0425-1B': 'blk0-3',
             'OLMo-2-1124-7B': 'blk0-7', 'nanochat-d12': 'blk0-2'}
for model, ablate in ABLATIONS.items():
    ledger_fig(model, ablate)

In [ ]:
# nanochat-d12: the late layers 9/10/11 separately and together, baseline vs the two
# early-block ablations.
model = 'nanochat-d12'
ledger_grid(model,
            [('blk9', [9]), ('blk10', [10]), ('blk11', [11]), ('blk9-11', [9, 10, 11])],
            [('baseline', CFG(model)), ('− blk0-2', 'ablate_blk0-2'), ('− blk3-5', 'ablate_blk3-5')],
            f'Ledger contributions of late layers — {model}, baseline vs early-block ablations',
            f'ledger_ablation_layers/{model}_late_layers', figsize=(16, 14))

In [ ]:
# The per-block depth stacks (the experiments.ipynb ledger stacks), baseline vs ablated,
# per ledger term. The embedding entropy line only makes sense on the ΔS stack.
TERM_LABEL = {'delta_s': 'signed ΔS', 'quality': 'quality', 'interference': 'interference'}

def ledger_term_stacks(mdl, ablate, terms=('delta_s', 'quality', 'interference')):
    srcs = lambda cfg: [(cfg, (f'blk{l}', 'block_ledger'), f'blk {l}') for l in range(NB[mdl])]
    emb = lambda cfg: (cfg, ('blk0.attn.in', 'acts_centered'), 'embedding', 'matrix_entropy')
    for term in terms:
        grid_start(ncols=2, sharey=True,
                   title=f'{mdl} — rank-entropy ledger ({TERM_LABEL[term]}), baseline vs − {ablate}')
        for ttl, cfg in (('baseline', CFG(mdl)), (f'− {ablate}', f'ablate_{ablate}')):
            plot_layer_contribution(mdl, srcs(cfg), yvar=term, normalize=False, title=ttl,
                                    alpha=0.85,
                                    **({'baseline_src': emb(cfg)} if term == 'delta_s' else {}))
        grid_show()

ledger_term_stacks('nanochat-d12', 'blk3-5')

In [ ]:
# pythia-1b: the same three depth stacks, baseline vs the blk3 (carrier) ablation.
ledger_term_stacks('pythia-1b-deduped', 'blk3')

In [ ]:
for model, nb in NB.items():
    ledger_term_stacks(model, QUARTERS[nb][0])

## OLMo-2: single mid-late layer + third quarter, baseline vs second-quarter ablation

In [ ]:
def ledger_pair(model, single):
    """2x2 ledger: rows = single layer / third quarter, cols = baseline / − second quarter."""
    L = NB[model]
    q2 = f'blk{L // 4}-{L // 2 - 1}'
    q3 = list(range(L // 2, 3 * L // 4))
    rows = [(f'blk{single}', [single]), (f'third quarter (blk{q3[0]}-{q3[-1]})', q3)]
    ledger_grid(model, rows, [('baseline', CFG(model)), (f'− {q2}', f'ablate_{q2}')],
                f'Ledger contributions — {model}, baseline vs {q2} writes zeroed',
                f'ledger_ablation_layers/{model}_q2_ablation_single_q3', figsize=(13, 8))

In [ ]:
ledger_pair('OLMo-2-0425-1B', 10)

In [ ]:
ledger_pair('OLMo-2-1124-7B', 16)

## OLMo-2: the middle four layers, no ablation

Both models in one figure, each panel the baseline ledger stack summed over that model's
four central blocks (1B: blk6-9, 7B: blk14-17).

In [ ]:
OLMO = ['OLMo-2-0425-1B', 'OLMo-2-1124-7B']
mid4 = lambda L: list(range(L // 2 - 2, L // 2 + 2))     # the four blocks straddling mid-depth
not_mid4 = lambda L : mid4(L) if L==16 else list(range(7,16))
grid_start(ncols=len(OLMO), figsize=(13, 4.5), sharex=True, emb=False, zero_line=True,
           title='Ledger contributions of the middle four layers — OLMo-2, no ablation',
           savedir='ledger_ablation_layers/olmo_middle4')
for j, model in enumerate(OLMO):
    layers = not_mid4(NB[model])
    plot_ledger_stack(model, CFG(model), blocks=layers,
                      title=f'{model} — blk{layers[0]}-{layers[-1]}',
                      ylabel='rank entropy' if j == 0 else None, legend=7 if j == 0 else None)
grid_show()

In [ ]:
OLMO = ['OLMo-2-0425-1B', 'OLMo-2-1124-7B']
first_half = lambda L: list(range(L // 2))     # the four blocks straddling mid-depth
first_threequarters = lambda L: list(range(L // 4 * 3))
# not_mid4 = lambda L : mid4(L) if L==16 else list(range(7,16))
grid_start(ncols=len(OLMO), figsize=(13, 4.5), sharex=True, emb=False, emb_delta=True,
           exp_axis=True, zero_line=True,
           title=r'$\Delta S$ contributions of non-final layers',
           savedir='ledger_ablation_layers1/olmo_b4_middle4')
for j, model in enumerate(OLMO):
    layers = first_threequarters(NB[model])
    plot_ledger_stack(model, CFG(model), blocks=layers,
                      title=f'{model.replace('1124-', '-').replace('0425-', '-').replace('-', ' ')}      Layers {layers[0]+1}-{layers[-1]+1}',
                      ylabel=r'$\Delta S$' if j == 0 else None, legend=7 if j == 0 else None)
grid_show()

#TODO plot trajectory from peak adjusted for each component

## pythia-1b: packed vs content-only tokens, baseline vs the L4 ablation

Full depth, so the Δ-embedding band and the e^ΔS ratio axis both read as the whole
model's rank budget. The packed sweep with and without L4's writes (blk3 in code indexing —
pythia-1b's carrier), then the same baseline on interior content tokens only
(data/results/content_tokens_full). There is no ablated content-token run, so the ablation
and the token filter are read one at a time, against the shared packed baseline.

In [ ]:
PY1B, ABL = 'pythia-1b-deduped', 'blk3'            # blk3 = L4 in the thesis' 1-indexed blocks
CONTENT = 'content_tokens_full'
# Three panels, one row: there is no ablated content-token run, so the ablation is read off the
# packed pair and the content sweep only says whether the baseline ledger survives the token filter.
PANELS = [('packed — baseline', CFG(PY1B)), ('packed — − L4', f'ablate_{ABL}'),
          ('content tokens — baseline', CONTENT)]
grid_start(ncols=3, sharey=True, pad=1.6, emb_delta=True, exp_axis=True, zero_line=True,
           title='pythia-1b ledger over full depth — packed vs content tokens, baseline vs − L4',
           savedir='ledger_ablation_layers/py1b_content_L4')
for j, (label, cfg) in enumerate(PANELS):
    plot_ledger_stack(PY1B, cfg, n_blocks=NB[PY1B], title=label,
                      ylabel='rank entropy' if j == 0 else None, legend=8 if j == 0 else None)
grid_show()

## All six models, full depth: packed baseline vs the content-token sweep

One figure per model, the whole ledger summed over every block — so the black ΔS total is
the measured depth differential S(before_final_norm) − S(embedding), and the brown ratio
axis is what that differential does to effective rank. Same panel as showcase §3, plus the
Δ-embedding band and the e^ΔS axis.

In [ ]:
ALL6 = ['pythia-410m-deduped', 'pythia-1b-deduped', 'pythia-6.9b-deduped',
        'nanochat-d12', 'OLMo-2-0425-1B', 'OLMo-2-1124-7B']

for model in ALL6:
    grid_start(ncols=2, emb_delta=True, exp_axis=True, zero_line=True,
               title=f'Ledger contributions, packed vs content tokens — {get_model_label(model)}',
               savedir=f'ledger_ablation_layers/baseline_vs_content_{model}')
    plot_ledger_stack(model, CFG(model), NB[model], title='packed')
    plot_ledger_stack(model, CONTENT, NB[model], title='content tokens')
    grid_show()